# unify_pums.ipynb

This notebook reads in the household and person-level PUMS for a given year, merges them, cleans up the ORIGIN/CHOSEN fields, and injects fields that will be used later on during modeling.

In [1]:
import os
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath("../.."))
from lib import io as lio


In [2]:
sample_size = 0.5  # proportion of people in PUMS to consider
year = 2018
directory = "us"
base_household_name = "psam_hus"
base_person_name = "psam_pus"

In [3]:
person_cols = set(
    pd.read_csv(f"{directory}/{base_person_name}a_{year}.csv", nrows=0).columns
)
household_cols = set(
    pd.read_csv(f"{directory}/{base_household_name}a_{year}.csv", nrows=0).columns
)

In [4]:
for col in household_cols:
    print(col)

FMHP
SERIALNO
REFR
RT
WGTP25
RNTP
HHT
FACCESSP
OCPIP
HHL
R18
WGTP22
WGTP53
WGTP59
FMRGP
WGTP38
ADJINC
MV
FVACSP
OTHSVCEX
FTAXP
WGTP43
WGTP73
WGTP74
WGTP28
GASP
FSTOVP
WGTP18
WGTP55
BDSP
FPLMPRP
WGTP62
WGTP16
GASFP
GRPIP
FREFRP
FFINCP
FRNTP
FDIALUPP
SMARTPHONE
R65
FACRP
FES
FHOTWATP
WGTP33
WGTP64
WGTP41
HINCP
WGTP65
PLM
FGASP
FTENP
VALP
COMPOTHX
MRGI
WGTP2
FSINKP
FBATHP
FINSP
FS
WGTP20
WGTP5
WGTP46
ACCESS
WGTP36
WGTP13
WGTP54
NPF
FFSP
WIF
HUGCL
MRGX
WGTP75
VACS
AGS
SATELLITE
WATP
WGTP71
MRGP
WGTP19
NOC
WGTP12
WGTP50
CONP
FHISPEEDP
WGTP15
WGTP37
R60
HFL
WGTP47
WGTP39
WATFP
WGTP45
FCOMPOTHXP
TAXAMT
FULP
PARTNER
GRNTP
FVEHP
WGTP24
WGTP27
FKITP
LNGI
INSP
WGTP11
KIT
SINK
TEL
WGTP52
WGTP80
WGTP30
WGTP14
WGTP76
BROADBND
WGTP70
NRC
SRNT
WGTP77
WGTP69
RNTM
RESMODE
RWATPR
WGTP4
WGTP58
WGTP23
FBROADBNDP
FSMXHP
WGTP8
TEN
DIALUP
SMP
HHLANP
FPARC
WGTP29
ADJHSG
SSMC
TABLET
FMVP
MRGT
FPLMP
FYBLP
ST
FHINCP
WORKSTAT
WGTP21
WGTP34
WGTP48
FGRNTP
RMSP
FSMARTPHONP
FULFP
WGTP17
WGTP61
FRMSP
FTELP
WGTP32
FRWAT

In [5]:
def filter_person_cols(col: str):
    # weight column
    if col.startswith("PWGTP"):
        return False
    # flag column
    if col[1:-1] in person_cols:
        return False
    return True


target_household_cols = {
    "SERIALNO",
    "NP",
    "TYPE",
    "TEN",
    "VALP",
    "VEH",
    "FES",
    "FINCP",
    "FPARC",
    "GRNTP",
    "GRPIP",
    "HHT",
    "HINCP",
    "OCPIP",
    "PARTNER",
    "R18",
    "SMOCP",
    "TAXAMT",
    "WIF",
    "HUPAOC",
    "HUPARC",
    "MULTG",
    "MV",
    "R65",
    "ACR",
    "MRGP",
    "MRGT",
    "NOC",
    "WKEXREL",
    "WORKSTAT",
}


def filter_household_cols(col: str):
    # weight column
    if col.startswith("WGTP"):
        return False
    # flag column
    if col[1:-1] in person_cols:
        return False
    return col in target_household_cols

In [6]:
# reading in the individual PUMS
dfs = []
dfs.append(
    pd.read_csv(
        f"{directory}/{base_person_name}a_{year}.csv", usecols=filter_person_cols
    )
)
dfs.append(
    pd.read_csv(
        f"{directory}/{base_person_name}b_{year}.csv", usecols=filter_person_cols
    )
)
df_p = pd.concat(dfs).reset_index(drop=True)

del dfs

In [7]:
for col in df_p.columns:
    print(col)

RT
SERIALNO
DIVISION
SPORDER
PUMA
REGION
ST
ADJINC
AGEP
CIT
CITWP
COW
DDRS
DEAR
DEYE
DOUT
DPHY
DRAT
DRATX
DREM
ENG
FER
GCL
GCM
GCR
HINS1
HINS2
HINS3
HINS4
HINS5
HINS6
HINS7
INTP
JWMNP
JWRIP
JWTR
LANX
MAR
MARHD
MARHM
MARHT
MARHW
MARHYP
MIG
MIL
MLPA
MLPB
MLPCD
MLPE
MLPFG
MLPH
MLPI
MLPJ
MLPK
NWAB
NWAV
NWLA
NWLK
NWRE
OIP
PAP
RELP
RETP
SCH
SCHG
SCHL
SEMP
SEX
SSIP
SSP
WAGP
WKHP
WKL
WKW
WRK
YOEP
ANC
ANC1P
ANC2P
DECADE
DIS
DRIVESP
ESP
ESR
FOD1P
FOD2P
HICOV
HISP
INDP
JWAP
JWDP
LANP
MIGPUMA
MIGSP
MSP
NAICSP
NATIVITY
NOP
OC
OCCP
PAOC
PERNP
PINCP
POBP
POVPIP
POWPUMA
POWSP
PRIVCOV
PUBCOV
QTRBIR
RAC1P
RAC2P
RAC3P
RACAIAN
RACASN
RACBLK
RACNH
RACNUM
RACPI
RACSOR
RACWHT
RC
SCIENGP
SCIENGRLP
SFN
SFR
VPS
WAOB
FAGEP
FCITWP
FFODP
FHISP
FINDP
FINTP
FJWDP
FJWMNP
FJWRIP
FLANP
FMARHYP
FMIGSP
FMILPP
FMILSP
FOCCP
FOIP
FPAP
FPERNP
FPINCP
FPOBP
FPOWSP
FRACP
FRELP
FRETP
FSEMP
FSSIP
FSSP
FWAGP
FWKHP
FYOEP


In [8]:
# reading in the household PUMS for referencing purposes
dfs = []
dfs.append(
    pd.read_csv(
        f"{directory}/{base_household_name}a_{year}.csv", usecols=filter_household_cols
    )
)
dfs.append(
    pd.read_csv(
        f"{directory}/{base_household_name}b_{year}.csv", usecols=filter_household_cols
    )
)

df_h = pd.concat(dfs).reset_index(drop=True)

del dfs

In [9]:
# cleaning missing identifier data
df_p["MIGPUMA"] = df_p["MIGPUMA"].fillna(0)
df_p["MIGSP"] = df_p["MIGSP"].fillna(0)

In [10]:
# filtering out the PUMS to people 18+ who moved from places in the contiguous united states to other places in the contiguous united states
mask = (
    (df_p["AGEP"] >= 18)
    & (df_p["MIGSP"] <= 56)
    & (~df_p["MIGSP"].isin([2, 15]))  # 2 and 15 correspond to Alaska and Hawaii
    & (df_p["ST"] <= 56)
    & (~df_p["ST"].isin([2, 15]))
)
print(df_p.shape)
df_subset = df_p.loc[mask].copy()
print(df_subset.shape)

(3214539, 158)
(2530899, 158)


In [11]:
# origin is the MIGSP + MIGPUMA
# need ints since it is treated as a float by default
origin = df_subset["MIGSP"].astype(int).astype(str).str.zfill(2) + df_subset[
    "MIGPUMA"
].astype(int).astype(str).str.zfill(5)
# chosen is the current location, ST + PUMA
chosen = df_subset["ST"].astype(int).astype(str).str.zfill(2) + df_subset[
    "PUMA"
].astype(int).astype(str).str.zfill(5)
# people who stayed had their origin is all zeros (due to fillna 0)
df_subset["STAY"] = np.where(origin == "0000000", 1, 0)

df_subset["ORIGIN"] = origin
df_subset["CHOSEN"] = chosen

In [12]:
puma_migpuma = lio.load_puma_migpuma("../geometry/equivalencies/puma_migpuma_2010.csv")
puma_migpuma.head()

,State,MIGPUMA
PUMA,,
0100100,01,0100190
0100200,01,0100290
0100301,01,0100290
0100302,01,0100290
0100400,01,0100400


In [13]:
df_subset["ORIGIN"].value_counts()

ORIGIN
0000000    2202915
0603700       8978
1703400       4911
2500390       4785
0400100       4617
            ...   
2201600         67
2101000         66
4806900         64
1702601         62
5401300         60
Name: count, Length: 976, dtype: int64

In [14]:
# backfill the stay origins to the MIGPUMA where they are currently (chosen == origin)
df_subset["ORIGIN"] = np.where(
    df_subset["STAY"] == 1,
    puma_migpuma.loc[df_subset["CHOSEN"], "MIGPUMA"],
    df_subset["ORIGIN"],
)
# fill in the origin state with this backfill in place
df_subset["ORIGIN_STATE"] = df_subset["ORIGIN"].str[:2]

In [15]:
df_subset["ORIGIN"].value_counts()

ORIGIN
0603700    81968
2500390    40213
1703400    33809
0400100    31999
0800190    28154
           ...  
2202100      625
2300600      623
4806900      607
0800400      605
2201600      598
Name: count, Length: 975, dtype: int64

In [16]:
df_subset["ORIGIN_STATE"].value_counts()

ORIGIN_STATE
06    299709
48    205061
12    164884
36    159419
42    104809
17    100946
39     94797
37     81176
26     79677
13     78773
34     71480
51     67055
53     59975
25     57059
04     54049
47     53598
18     52948
29     49511
24     47599
55     47508
27     44018
08     43874
45     39745
01     38268
21     35838
22     34730
41     33613
09     29379
40     28932
19     25447
05     23796
28     23002
20     22979
32     22592
49     22142
35     15304
31     15054
54     14819
16     12550
33     11295
23     10893
44      8509
30      8205
10      7475
46      6823
38      6195
11      5544
50      5332
56      4513
Name: count, dtype: int64

In [17]:
df_subset["CHOSEN"].value_counts()

CHOSEN
0102500    3609
5310200    3393
5500100    3256
5500700    3183
1200500    2778
           ... 
2701403     430
4804633     429
4203207     421
5541001     417
2701402     412
Name: count, Length: 2336, dtype: int64

In [ ]:
# taking sample of PUMS using consistent random seed
df = df_subset.sample(n=round(df_subset.shape[0] * sample_size), random_state=8470897)
df

,RT,SERIALNO,DIVISION,SPORDER,PUMA,REGION,ST,ADJINC,AGEP,CIT,...,FSEMP,FSSIP,FSSP,FWAGP,FWKHP,FYOEP,STAY,ORIGIN,CHOSEN,ORIGIN_STATE
2,P,2018GQ0000219,6,1,2000,3,1,1013097,53,1,...,0,0,0,1,0,0,1,0102090,0102000,01
5,P,2018GQ0000390,6,1,2400,3,1,1013097,30,1,...,0,0,0,0,0,0,1,0102400,0102400,01
8,P,2018GQ0001008,6,1,2200,3,1,1013097,41,1,...,0,0,0,0,0,0,1,0102200,0102200,01
9,P,2018GQ0001011,6,1,1600,3,1,1013097,18,1,...,0,0,0,0,0,0,0,0101900,0101600,01
11,P,2018GQ0001207,6,1,2400,3,1,1013097,79,1,...,0,0,1,0,0,0,1,0102400,0102400,01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3214529,P,2018HU1399964,8,1,500,4,56,1013097,55,1,...,0,0,0,0,0,0,1,5600500,5600500,56
3214530,P,2018HU1399964,8,2,500,4,56,1013097,77,1,...,0,0,0,0,0,0,1,5600500,5600500,56
3214532,P,2018HU1400326,8,2,400,4,56,1013097,34,1,...,1,1,1,1,1,0,1,5600400,5600400,56
3214536,P,2018HU1400502,8,1,100,4,56,1013097,49,1,...,0,0,0,0,0,0,1,5600100,5600100,56


In [19]:
# should not get duplicate measurements for the same SERIALNO
assert df_h["SERIALNO"].value_counts().max() == 1
# merge the household info into df
df = pd.merge(df, df_h, left_on="SERIALNO", right_on="SERIALNO", how="left")

In [20]:
# own child
df["CHILD_UNDER_6"] = np.where(df["HUPAOC"] == 1, 1, 0)
df["CHILD_6_TO_17"] = np.where(df["HUPAOC"] == 2, 1, 0)
df["CHILD"] = np.where(df["HUPAOC"].isin([1, 2, 3]), 1, 0)
df["CHILD"].value_counts()

CHILD
0    950260
1    315189
Name: count, dtype: int64

In [21]:
df["WORK2_MAR"] = np.where(df["FES"] == 1, 1, 0)
df["WORK1_MAR"] = np.where((df["FES"] <= 4) & (df["FES"] >= 2), 1, 0)
df["SINGLE_PARENT"] = np.where((df["HHT"] == 2) | (df["HHT"] == 3), 1, 0)

In [22]:
df["EDU_NOHIGH"] = np.where(df["SCHL"] <= 15, 1, 0)
df["EDU_HIGH_BUT_NOT_BACHELORS"] = np.where(
    (df["SCHL"] <= 20) & (df["SCHL"] >= 16), 1, 0
)
df["EDU_BACHELORS_OR_HIGHER"] = np.where(df["SCHL"] >= 21, 1, 0)
df["EDU_ONLY_HIGH"] = np.where(df["SCHL"].isin([16, 17]), 1, 0)
df["EDU_SOME_COLLEGE"] = np.where(df["SCHL"].isin([18, 19, 20]), 1, 0)
df["EDU_ONLY_BACHELORS"] = np.where(df["SCHL"] == 21, 1, 0)
df["EDU_GRADUATE_DEG"] = np.where(df["SCHL"] >= 22, 1, 0)
df["EDU_HAS_DEGREE"] = np.where(df["SCHL"] >= 21, 1, 0)
df["EDU_NO_DEGREE"] = np.where(df["SCHL"] <= 20, 1, 0)

In [23]:
df["AGE_UNDER_18"] = np.where(df["AGEP"] < 18, 1, 0)
df["AGE_18_34"] = np.where((df["AGEP"] <= 34) & (df["AGEP"] >= 18), 1, 0)
df["AGE_35_64"] = np.where((df["AGEP"] >= 35) & (df["AGEP"] <= 64), 1, 0)
df["AGE_18_22"] = np.where(df["AGEP"] <= 22, 1, 0)
df["AGE_23_29"] = np.where((df["AGEP"] >= 23) & (df["AGEP"] <= 29), 1, 0)
df["AGE_30_39"] = np.where((df["AGEP"] >= 30) & (df["AGEP"] <= 39), 1, 0)
df["AGE_40_49"] = np.where((df["AGEP"] >= 40) & (df["AGEP"] <= 49), 1, 0)
df["AGE_50_64"] = np.where((df["AGEP"] >= 50) & (df["AGEP"] <= 64), 1, 0)
df["AGE_OVER_65"] = np.where((df["AGEP"] >= 65), 1, 0)
df["FOREIGN"] = np.where(df["NATIVITY"] == 2, 1, 0)

In [24]:
df["IN_COLLEGE"] = np.where((df["SCHG"] == 15) | (df["SCHG"] == 16), 1, 0)

In [25]:
df["WOMAN_WITH_CHILD"] = np.where((df["PAOC"] >= 1) & (df["PAOC"] <= 3), 1, 0)
df["MALE"] = np.where(df["SEX"] == 1, 1, 0)
df["FEMALE"] = np.where(df["SEX"] == 0, 1, 0)

In [26]:
df["MARRIED"] = np.where(df["MAR"] == 1, 1, 0)
df["RECENTLY_WIDOWED_OR_DIVORCED"] = np.where(
    (df["MARHD"] == 1) | (df["MARHW"] == 1), 1, 0
)
df["RECENTLY_MARRIED"] = np.where(df["MARHM"] == 1, 1, 0)
df["MARRIED_MORE_THAN_YEAR"] = np.where(df["MARRIED"] & ~df["RECENTLY_MARRIED"], 1, 0)

In [27]:
df["IN_MILITARY"] = np.where(df["MIL"] == 1, 1, 0)
df["UNEMPLOYED"] = np.where(df["ESR"] == 3, 1, 0)
df["NOT_IN_LABOR_FORCE"] = np.where(df["ESR"] == 6, 1, 0)
df["IN_LABOR_FORCE"] = np.where(df["ESR"] == 6, 0, 1)

In [28]:
df["WHITE"] = np.where(df["RAC1P"] == 1, 1, 0)
df["BLACK"] = np.where(df["RAC1P"] == 2, 1, 0)
df["INDIAN"] = np.where(df["RAC1P"].isin([3, 4, 5]), 1, 0)
df["AAPI"] = np.where(df["RAC1P"].isin([6, 7]), 1, 0)
df["OTHER_RACE"] = np.where(df["RAC1P"].isin([8, 9]), 1, 0)
df["LATINO"] = np.where(df["HISP"] != 1, 1, 0)
# disable other races to unify with the ACS notion of total population not hispanic of latino american
for col in ["WHITE", "BLACK", "INDIAN", "AAPI", "OTHER_RACE"]:
    df[col] = np.where(df["LATINO"] == 1, 0, df[col])

# make another column making latino a category among races
df["RACE_ETHNICITY"] = np.where(df["LATINO"] == 1, 99, df["RAC1P"])

In [29]:
df.shape

(1265449, 235)

In [ ]:
df.to_parquet(f"pums_{sample_size * 100:.0f}_{year}.parquet")

In [31]:
# people who've recently had children category
# NOTE: this is a little iffy since this only applies to the women

# def add_recent_child_flag(df: pd.DataFrame) -> pd.DataFrame:
#     """FER_CL cleaning and inference for whether a household recently had a child."""
#     df["FER_CL"] = df["FER"].fillna(0)
#     df["FER_CL"] = np.where(df["FER_CL"] == 2, 0, df["FER_CL"])
#     rec_child = df.groupby("SERIALNO")["FER_CL"].max()
#     df["REC_CHILD"] = rec_child.loc[df["SERIALNO"]].values
#     return df
# df = lclean.add_recent_child_flag(df)
# df["FER_CL"].value_counts()